In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

english = open(r"D:\Jupyter\NLP Skills\Hometasks\Attention-Based English–Tamil\En-Ta English.txt", encoding="utf-8").read().splitlines()[3:]
tamil = open(r"D:\Jupyter\NLP Skills\Hometasks\Attention-Based English–Tamil\En-Ta Tamil.txt", encoding="utf-8").read().splitlines()[3:]

df = pd.DataFrame({"English": english, "Tamil": tamil})
df = df.dropna()
df = df[(df.English.str.strip() != "") & (df.Tamil.str.strip() != "")]

# Use a smaller subset for faster training
np.random.seed(42)
idx = np.random.permutation(len(df))[:5000]
df = df.iloc[idx]

# Tokenization
def tokenize(text):
    return text.lower().split()

# Vocabulary
def build_vocab(sentences):
    vocab = {"<pad>": 0, "<unk>": 1, "<sos>": 2, "<eos>": 3}
    for s in sentences:
        for word in tokenize(s):
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

en_vocab = build_vocab(df.English)
ta_vocab = build_vocab(df.Tamil)

# Convert sentences to numbers
MAX_LEN = 20

def encode(sentence, vocab):
    words = tokenize(sentence)[:MAX_LEN-2]
    ids = [vocab["<sos>"]]
    ids += [vocab.get(w, vocab["<unk>"]) for w in words]
    ids += [vocab["<eos>"]]
    ids += [vocab["<pad>"]] * (MAX_LEN - len(ids))
    return ids[:MAX_LEN]

X = torch.tensor([encode(s, en_vocab) for s in df.English])
Y = torch.tensor([encode(s, ta_vocab) for s in df.Tamil])

loader = DataLoader(TensorDataset(X, Y), batch_size=32, shuffle=True)

# Encoder
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb=64, hidden=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb)
        self.gru = nn.GRU(emb, hidden, batch_first=True)

    def forward(self, x):
        x = self.embedding(x)
        return self.gru(x)

# Attention
class Attention(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn = nn.Linear(hidden * 2, hidden)
        self.v = nn.Linear(hidden, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        h = hidden[-1].unsqueeze(1).repeat(1, encoder_outputs.size(1), 1)
        energy = torch.tanh(self.attn(torch.cat((h, encoder_outputs), dim=2)))
        scores = self.v(energy).squeeze(2)
        return torch.softmax(scores, dim=1)

# Decoder
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb=64, hidden=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb)
        self.attention = Attention(hidden)
        self.gru = nn.GRU(emb + hidden, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, x, hidden, encoder_outputs):
        x = self.embedding(x).unsqueeze(1)
        attn = self.attention(hidden, encoder_outputs).unsqueeze(1)
        context = torch.bmm(attn, encoder_outputs)
        x = torch.cat((x, context), dim=2)
        output, hidden = self.gru(x, hidden)
        return self.fc(output.squeeze(1)), hidden

encoder = Encoder(len(en_vocab))
decoder = Decoder(len(ta_vocab))

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=0.001
)

# Training
for epoch in range(15):
    total_loss = 0

    for x, y in loader:
        optimizer.zero_grad()

        encoder_outputs, hidden = encoder(x)

        decoder_input = y[:, 0]
        loss = 0

        for t in range(1, MAX_LEN):
            output, hidden = decoder(
                decoder_input, hidden, encoder_outputs
            )
            loss += criterion(output, y[:, t])
            decoder_input = y[:, t]

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss/len(loader):.4f}")

# English -> Tamil translation
def translate(sentence):
    x = torch.tensor([encode(sentence, en_vocab)])

    with torch.no_grad():
        encoder_outputs, hidden = encoder(x)

    word = torch.tensor([ta_vocab["<sos>"]])
    result = []

    for _ in range(MAX_LEN):
        with torch.no_grad():
            output, hidden = decoder(word, hidden, encoder_outputs)

        word = output.argmax(1)

        if word.item() == ta_vocab["<eos>"]:
            break

        for w, i in ta_vocab.items():
            if i == word.item():
                if w not in ["<pad>", "<sos>", "<eos>"]:
                    result.append(w)
                break

    return " ".join(result)

# User input
while True:
    sentence = input("\nEnter English sentence (or type exit): ")
    if sentence.lower() == "exit":
        break
    print("Tamil Translation:", translate(sentence))